In [0]:

%sql
--create catalog bronze 

--describe schema HR_POC.silver


--ALTER SCHEMA HR_POC.silver
--SET MANAGED LOCATION 'abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/';


In [0]:
# employee_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Employee/employee_20260901.csv")
# )


# department_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Department/department_20260901.csv")
# )

# designation_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Designation/Designation_20260901.csv")
# )

# salary_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Salary/salary_20260901.csv")
# )

# attendance_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Attendence/attendence_20260901.csv")
# )

# attrition_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/Attrition/attrition_20260901.csv")
# )





In [0]:
from datetime import date
from pyspark.sql import DataFrame


DATASETS = [
    {"name": "employee", "folder": "Employee", "file_prefix": "employee"},
    {"name": "designation", "folder": "Designation", "file_prefix": "designation"},
    {"name": "department", "folder": "Department", "file_prefix": "department"},
    {"name": "salary", "folder": "Salary", "file_prefix": "salary"},
    {"name": "attendance", "folder": "Attendence", "file_prefix": "attendence"},
    {"name": "attrition", "folder": "Attrition", "file_prefix": "attrition"}   
]

BASE_PATH = "abfss://hr-data@hranalyticspoc2026.dfs.core.windows.net/Bronze/"

def read_all_datasets() -> dict:
    """
    Read all datasets and return as dictionary of DataFrames
    """
    today_str = date.today().strftime("%Y%m%d")
    dataframes = {}
    
    for dataset in DATASETS:
        path = f"{BASE_PATH}{dataset['folder']}/{dataset['file_prefix']}_{today_str}.csv"
        
        try:
            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(path)
            )
            dataframes[dataset['name']] = df
            print(f"Successfully loaded {dataset['name']} dataset")
        except Exception as e:
            print(f"Error loading {dataset['name']}: {str(e)}")
            dataframes[dataset['name']] = None
    
    return dataframes


dfs = read_all_datasets()

employee_df = dfs["employee"]
designation_df = dfs["designation"]
department_df = dfs["department"]
salary_df = dfs["salary"]
attendance_df = dfs["attendance"]
attrition_df = dfs["attrition"]

In [0]:
print("Employee:", employee_df.count())
print("Department:", department_df.count())
print("designation:", designation_df.count())
print("Salary:", salary_df.count())
print("Attendance:", attendance_df.count())
print("Attrition:", attrition_df.count())

In [0]:
display(employee_df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, isnan


spark = SparkSession.builder.getOrCreate()


null_counts = employee_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in employee_df.columns
])

# employee_df.show()
null_counts.show()


In [0]:
employee_df.groupBy("gender") \
    .count() \
    .filter(col("count") > 1) \
    .show()

employee_clean = employee_df.dropDuplicates(["employee_id"])

employee_df.printSchema()

In [0]:
from pyspark.sql.functions import trim, initcap

employee_clean = (
    employee_clean
    .withColumn("employee_name", trim(col("employee_name")))
    .withColumn("location", initcap(trim(col("location"))))
    .withColumn("employment_type", initcap(trim(col("employment_type"))))
    .withColumn("status", initcap(trim(col("status"))))
)

valid_employee = employee_clean.filter(
    col("employee_id").isNotNull() &
    col("joining_date").isNotNull() &
    col("department_id").isNotNull() &
    col("status").isin("Active", "Inactive")
)

invalid_employee = employee_clean.subtract(valid_employee)

print("Valid:", valid_employee.count())
print("Invalid:", invalid_employee.count())


In [0]:

null_counts = salary_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in salary_df.columns
])

salary_df.show()
null_counts.show()

In [0]:
salary_df.groupBy("salary_grade") \
    .count() \
    .filter(col("count") > 1) \
    .show()

salary_clean = salary_df.dropDuplicates(["employee_id"])

salary_df.printSchema()

In [0]:
from pyspark.sql.functions import to_date, col
salary_clean = (
    salary_df
    .dropDuplicates(["employee_id", "effective_date"])
    .withColumn("effective_date", col("effective_date").try_cast("date"))
    .withColumn("salary", col("salary").cast("decimal(12,2)"))
    .withColumn("bonus", col("bonus").cast("decimal(12,2)"))
)

valid_salary = salary_clean.filter(
    col("salary").isNotNull() &
    col("salary_grade").isNotNull()
)

invalid_salary = salary_clean.filter(
    col("salary").isNull() | (col("salary") <= 0)
)

print("Valid:", valid_salary.count())
print("Invalid:", invalid_salary.count())


display(valid_salary)


In [0]:


department_clean = (
    department_df
    .dropDuplicates(["department_id"])
    .withColumn("department_name", trim(col("department_name")))
    .withColumn("business_unit", trim(col("business_unit")))
    .withColumn("location", initcap(trim(col("location"))))
)

attendance_clean = (
    attendance_df
    .dropDuplicates(["employee_id", "attendance_date"])
    .withColumn("attendance_date", to_date("attendance_date", "M/d/yyyy"))
    .withColumn("working_hours", col("working_hours").cast("decimal(5,2)"))
    .withColumn("overtime_hours", col("overtime_hours").cast("decimal(5,2)"))
)

invalid_attendance = attendance_clean.filter(
    (col("working_hours") < 0) |
    (col("overtime_hours") < 0)
)


attrition_clean = (
    attrition_df
    .dropDuplicates(["employee_id"])
    .withColumn("exit_date", to_date("exit_date"))
    .withColumn("exit_reason", trim(col("exit_reason")))
    .withColumn("exit_type", initcap(trim(col("exit_type"))))
)

designation_clean = (
    designation_df
    .dropDuplicates(["Designation_id"])
    .withColumn("Designation_title", trim(col("Designation_title")))
    .withColumn("Designation_level", trim(col("Designation_level")))
    .withColumn("Designation_category", trim(col("Designation_category")))
)
from pyspark.sql.functions import to_date
attrition_clean = (
    attrition_df
    .dropDuplicates(["employee_id"])
    .withColumn("exit_date", to_date("exit_date"))
    .withColumn("exit_reason", trim(col("exit_reason")))
    .withColumn("exit_type", initcap(trim(col("exit_type"))))
)

In [0]:
employee_clean.join(
    department_clean,
    "department_id",
    "left_anti"
).show()

employee_clean.join(
    designation_clean,
    "Designation_id",
    "left_anti"
).show()

In [0]:
# spark.sql("DESCRIBE SCHEMA hr_poc.silver")

#spark.sql("CREATE SCHEMA IF NOT EXISTS hr_poc.silver")

(
    valid_employee
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.employee")
)

(
    department_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.department")
)

(
    designation_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.designation")
)

(
    valid_salary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.salary")
)

(
    attendance_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.attendance")
)

(
    attrition_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hr_poc.silver.attrition")
)


In [0]:
%sql
SELECT count(*) FROM hr_poc.silver.attrition;
SELECT count(*) FROM hr_poc.silver.employee;
SELECT count(*) FROM hr_poc.silver.attendance;
SELECT count(*) FROM hr_poc.silver.salary;
SELECT count(*) FROM hr_poc.silver.department;
SELECT count(*) FROM hr_poc.silver.designation;



In [0]:
%sql
Describe extended hr_poc.silver.employee

In [0]:
%sql
alter table hr_poc.silver.employee set tblproperties ('delta.enableDeletionVectors' = 'false')

In [0]:

dbutils.fs.mkdirs("/Volumes/hr_poc/gold/hr/salary")

In [0]:
%sql
    GRANT CREATE VOLUME ON SCHEMA hr_poc TO `abhi.kaushik.bharti@gmail.com`;
GRANT USE SCHEMA ON SCHEMA main.default TO `abhi.kaushik.bharti@gmail.com`;
-- GRANT USE CATALOG ON CATALOG main TO 'abhi.kaushik.bharti@gmail.com';

In [0]:
# ─────────────────────────────────────────────
# 1. TEXT  – free-form string input
# ─────────────────────────────────────────────
dbutils.widgets.text("text_widget", "default_value", "Text Widget")
text_val = dbutils.widgets.get("text_widget")
print(f"Text value: {text_val}")

# ─────────────────────────────────────────────
# 2. DROPDOWN  – pick one from a fixed list
# ─────────────────────────────────────────────
dbutils.widgets.dropdown("dropdown_widget", "Option1",
                         ["Option1", "Option2", "Option3"],
                         "Dropdown Widget")
dropdown_val = dbutils.widgets.get("dropdown_widget")
print(f"Dropdown value: {dropdown_val}")

# ─────────────────────────────────────────────
# 3. COMBOBOX  – dropdown + free-text entry
# ─────────────────────────────────────────────
dbutils.widgets.combobox("combobox_widget", "Option1",
                         ["Option1", "Option2", "Option3"],
                         "Combobox Widget")
combobox_val = dbutils.widgets.get("combobox_widget")
print(f"Combobox value: {combobox_val}")

# ─────────────────────────────────────────────
# 4. MULTISELECT  – pick one or more values
# ─────────────────────────────────────────────
dbutils.widgets.multiselect("multiselect_widget", "Option1",
                            ["Option1", "Option2", "Option3"],
                            "Multiselect Widget")
multiselect_val = dbutils.widgets.get("multiselect_widget")
print(f"Multiselect value: {multiselect_val}")
# multiselect returns a comma-separated string; split to get a list:
selected_list = multiselect_val.split(",")
print(f"Multiselect list: {selected_list}")

# ─────────────────────────────────────────────
# 5. Retrieve ALL widget values at once
# ─────────────────────────────────────────────
all_widgets = dbutils.widgets.getAll()
print(f"All widget values: {all_widgets}")

# ─────────────────────────────────────────────
# 6. Remove a specific widget / remove all
# ─────────────────────────────────────────────
# dbutils.widgets.remove("text_widget")   # remove one
# dbutils.widgets.removeAll()             # remove all